In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("scam_messages_train.csv")

feature_cols = [
    "text",
    "message_type",
    "platform",
    "job_type",
    "contains_link",
    "contains_email",
    "has_payment_request",
    "asks_personal_info"
]

X = df[feature_cols]
y_label = df["label"]
y_type = df["scam_type"]

X_train, X_test, y_train_label, y_test_label, y_train_type, y_test_type = train_test_split(
    X, y_label, y_type,
    test_size=0.2,
    random_state=42,
    stratify=y_label
)

text_col = "text"
cat_cols = ["message_type", "platform", "job_type"]
bin_cols = ["contains_link", "contains_email", "has_payment_request", "asks_personal_info"]

preprocessor_label = ColumnTransformer([
    ("text", TfidfVectorizer(max_features=5000, ngram_range=(1, 2)), text_col),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("bin", "passthrough", bin_cols)
])

preprocessor_type = ColumnTransformer([
    ("text", TfidfVectorizer(max_features=5000, ngram_range=(1, 2)), text_col),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("bin", "passthrough", bin_cols)
])

label_model = Pipeline([
    ("prep", preprocessor_label),
    ("clf", LogisticRegression(max_iter=1000))
])

type_model = Pipeline([
    ("prep", preprocessor_type),
    ("clf", LogisticRegression(max_iter=1000))
])

label_model.fit(X_train, y_train_label)
type_model.fit(X_train, y_train_type)

y_pred_label = label_model.predict(X_test)
y_pred_type = type_model.predict(X_test)

print("Label model accuracy:", accuracy_score(y_test_label, y_pred_label))
print("Type model accuracy:", accuracy_score(y_test_type, y_pred_type))

Label model accuracy: 1.0
Type model accuracy: 0.619625
